In [ ]:
# **Author: Shakir Showkat Sofi**
!pip install git+https://github.com/PGelss/scikit_tt.git

In [2]:
import numpy as np
import scipy as sp
from scikit_tt import tensor_train as tt
import time

## Utility functions

In [3]:
def block2right(X, n, tol=1e-8):
    """
    Shift block index from core n to core n+1 in a Block-TT-Matrix.
    - X.cores[n]   : (rL, In, K, rR)      # block core
    - X.cores[n+1] : (rR, Inext, 1, rNext)

    Output:
      - core[n]     : (rL, In, 1, rRnew)
      - core[n+1]   : (rRnew, Inext, K, rNext) # block core
    """
    cores = list(X.cores)
    coren = cores[n]
    corenp1 = cores[n+1]

    rL, In, K, rR = coren.shape
    rR_chk, Inext, Jnext, rNext = corenp1.shape
    assert rR == rR_chk, "Rank mismatch between core n and n+1"
    assert Jnext == 1, "Expected singleton physical index in non-block core"

    # reshape (rL*In) x (K*rR)
    M = coren.reshape(rL * In, K * rR)
    Q, R = sp.linalg.qr(M, mode='economic', overwrite_a=True)
    rRnew = Q.shape[1]  # use tol for truncation

    # new ordinary TT-matrix core at n: (rL, In, 1, rRnew)
    coren_new = Q.reshape(rL, In, 1, rRnew)

    # new block core at n+1
    # R: (rRnew, K, rR), corenp1: (rR, Inext, 1, rNext)
    R_sh = R.reshape(rRnew, K, rR)
    corenp1_new = np.einsum('akb,bid->aikd', R_sh, corenp1[:,:,0,:], optimize=True)
    # result: (rRnew, Inext, K, rNext)

    # rebuild core tuple
    newcores = cores[:n] + [coren_new] + [corenp1_new] + cores[n+2:]
    return tt.TT(newcores)

def block2left(X, n, tol=1e-8):
    cores = list(X.cores)
    coren = cores[n]
    coren_prev = cores[n-1]

    rL, In, K, rR = coren.shape
    rPrev, Iprev, Jprev, rL_chk = coren_prev.shape
    assert rL == rL_chk, "Rank mismatch between core n-1 and n"
    assert Jprev == 1, "Expected singleton physical index in non-block core"

    # reshape M = (rL*K) x (In*rR)
    M = coren.transpose(0, 2, 1, 3).reshape(rL*K, In*rR)
    R, Q = sp.linalg.rq(M, mode='economic', overwrite_a=True,)
    rLnew = R.shape[1] # use tol for truncation

    # new block core at n-1: (rPrev, Iprev, K, rLnew)
    R_sh = R.reshape(rL, K, rLnew)
    coren_prev_new = np.einsum('pil,lka->pika', coren_prev[:,:,0,:], R_sh, optimize=True)

    # new ordinary core at n: (rLnew, In, 1, rR)
    coren_new = Q.reshape(rLnew, In, rR)
    coren_new = coren_new.reshape(rLnew, In, 1, rR)

    newcores = cores[:n-1] + [coren_prev_new] + [coren_new] + cores[n+1:]
    return tt.TT(newcores)

In [4]:
# Right interface products
def _right_interface(i: int, stack_R: list[np.ndarray], operator: tt.TT, U: tt.TT, V: tt.TT=None):
    if V is None:
        V = U.copy()
    if i == operator.order - 1:
        # last stack element is 1
        stack_R[i] = np.array([1], ndmin=3)
    else:
        # contract previous stack element with solution and operator cores
        stack_R[i] = np.tensordot(np.conj(U.cores[i + 1][:, :, 0, :]), stack_R[i + 1], axes=(2, 2))
        stack_R[i] = np.tensordot(operator.cores[i + 1], stack_R[i], axes=([1, 3], [1, 3]))
        stack_R[i] = np.tensordot(V.cores[i + 1][:, :, 0, :], stack_R[i], axes=([1, 2], [1, 3]))

# Left interface products
def _left_interface(i: int, stack_L: list[np.ndarray], operator: tt.TT, U: tt.TT, V: tt.TT=None):
    if V is None:
        V = U.copy()
    if i == 0:
        # first stack element is 1
        stack_L[i] = np.array([1], ndmin=3)
    else:
        # contract previous stack element with solution and operator cores
        stack_L[i] = np.tensordot(stack_L[i - 1], V.cores[i - 1][:, :, 0, :], axes=(0, 0))
        stack_L[i] = np.tensordot(stack_L[i], operator.cores[i - 1], axes=([0, 2], [0, 2]))
        stack_L[i] = np.tensordot(stack_L[i], np.conj(U.cores[i - 1][:, :, 0, :]), axes=([0, 2], [0, 1]))

# Effective operative: eff_op(mu) = contract(stack_L<mu, Op, stack_R>mu)
def _effective_op(i: int,        stack_L:  list[np.ndarray],
                                 stack_R: list[np.ndarray],
                                 operator: tt.TT, U: tt.TT, V: tt.TT=None) -> np.ndarray:
    if V is None:
        V = U.copy()
    # contract stack elements and operator core
    eff_op = np.tensordot(stack_L[i], operator.cores[i], axes=(1, 0))
    eff_op = np.tensordot(eff_op, stack_R[i], axes=(4, 1))

    # transpose and reshape micro matrix
    eff_op = eff_op.transpose([1, 2, 5, 0, 3, 4]).reshape(
        U.ranks[i] * operator.row_dims[i] * U.ranks[i + 1],
        V.ranks[i] * operator.col_dims[i] * V.ranks[i + 1])

    return eff_op

## SVD using DMRG

In [5]:
def hilbert_matrix(N): # test matrix
    m = 2**N
    i = np.arange(1, m+1).reshape(-1, 1)   # column vector
    j = np.arange(1, m+1).reshape(1, -1)   # row vector
    return 1.0 / (i + j - 1)

In [18]:
N = 8
D = [2]*N
P = [3]*N
# H = hilbert_matrix(N)
# Ht = H.reshape(D+D)
# H.shape, Ht.shape
# operator = tt.TT(Ht, threshold=1e-5)
ttrank = [1] + [3]*(N-1) + [1]
operator  = tt.rand(row_dims=D, col_dims=P, ranks=ttrank)
operator


Tensor train with order    = 9, 
                  row_dims = [2, 2, 2, 2, 2, 2, 2, 2, 2], 
                  col_dims = [3, 3, 3, 3, 3, 3, 3, 3, 3], 
                  ranks    = [1, 3, 3, 3, 3, 3, 3, 3, 3, 1]

In [19]:
K = 5
Opmat = operator.matricize()
tstart = time.time()
uo, so, vo = np.linalg.svd(Opmat)
tend = time.time()

print('Time taken: ', tend - tstart)

# Rank-K approximation error
approx_error = np.linalg.norm(Opmat - uo[:, :K] @ np.diag(so[:K]) @ vo[:K, :])
print('Top K singular values:', so[:K])
print('Approximation error:', approx_error)

Time taken:  130.16850423812866
Top K singular values: [57128.10560151 14563.22961611  8568.04834109  8419.40461359
  5775.33197752]
Approximation error: 10876.814986572575


DMRG for SVD

In [20]:
# Initialize  TT SVD
colshp = [1]*N
colshp[0] = K
ttrankU =  [1] + [int(np.ceil(K/x)) for x in operator.row_dims[:-1]] + [1]
ttrankV =  [1] + [int(np.ceil(K/x)) for x in operator.col_dims[:-1]] + [1]

Uinit = tt.rand(operator.row_dims, colshp, ranks=list(ttrankU)).ortho_right() # first core unnormalized
Vinit = tt.rand(operator.col_dims, colshp, ranks=list(ttrankV)).ortho_right() # first core unnormalized
Uinit, Vinit

(
 Tensor train with order    = 9, 
                   row_dims = [2, 2, 2, 2, 2, 2, 2, 2, 2], 
                   col_dims = [5, 1, 1, 1, 1, 1, 1, 1, 1], 
                   ranks    = [1, 3, 3, 3, 3, 3, 3, 3, 2, 1],
 
 Tensor train with order    = 9, 
                   row_dims = [3, 3, 3, 3, 3, 3, 3, 3, 3], 
                   col_dims = [5, 1, 1, 1, 1, 1, 1, 1, 1], 
                   ranks    = [1, 2, 2, 2, 2, 2, 2, 2, 2, 1])

For SVD, use `U` and `V`. Here, we only use `U`, which is EVD case.

In [23]:
orderswp = list(range(N-1)) + list(range(N-1, 0, -1))
U = Uinit.copy()
V = Vinit.copy()
Op = operator.copy()
order = U.order
nswps = 5
swp = 1
found = False
# initialize stacks
stack_L = [None] * order
stack_R = [None] * order
for i in reversed(range(N)):
    _right_interface(i, stack_R, Op, U, V)

# begin ALS
tstart  = time.time()
while swp<=nswps:
    incr=True # left to right sweep
    swp+=1
    for n in orderswp:
        if n==N-1:
            incr=False # right to left sweep

        # --- build left stack ---
        _left_interface(n, stack_L, Op, U, V)

        # --- build right stack ---
        _right_interface(n, stack_R, Op, U, V)

        # --- construct effective operator for n site ---
        eff_op = _effective_op(n, stack_L, stack_R, Op, U, V)

        Uop, Sop, Vop = np.linalg.svd(eff_op, full_matrices=True)
        K2 = np.minimum(K, Uop.shape[1])
        Ss = Sop[:K2]
        Us = Uop[:, :K2]
        Vs = Vop[:K2, :]

        # --- update n-th core of U and V ---
        U.cores[n] = Us.reshape(U.ranks[n], U.row_dims[n], U.ranks[n+1], K2).transpose(0, 1, 3, 2)
        V.cores[n] = Vs.reshape(K2, V.ranks[n], V.row_dims[n], V.ranks[n+1]).transpose(1, 2, 0, 3)

        # --- move index K ----
        if incr:
            U = block2right(U, n)
            V = block2right(V, n)
        else:
            U = block2left(U, n)
            V = block2left(V, n)

        sseu = np.max(sp.linalg.subspace_angles(U.matricize(), uo[:,:K]))
        ssev = np.max(sp.linalg.subspace_angles(V.matricize(), vo[:K,:].T))
        print('Subspace Error\n', n, sseu, ssev)

        mse = np.linalg.norm(U.matricize()@np.diag(Ss)@V.matricize().conj().T-Opmat)
        print('Norm Error\n', mse)

        if max(sseu, ssev) < 1e-7:
            found = True
            break
    if found:
        break

tend = time.time()
print('Time taken: ', tend-tstart)

Subspace Error
 0 1.5707963267948966 1.5707963267948966
Norm Error
 50658.17368153289
Subspace Error
 1 1.5707963267948966 1.5707963057214724
Norm Error
 49744.76377893891
Subspace Error
 2 1.5707962451778754 1.5707962730679959
Norm Error
 48697.64807789365
Subspace Error
 3 1.5707956827612624 1.570796181556343
Norm Error
 45297.970534134976
Subspace Error
 4 1.5707962690829476 1.5707962796732875
Norm Error
 40795.92930353725
Subspace Error
 5 1.570776968305311 1.5707813939130906
Norm Error
 36407.60472408523
Subspace Error
 6 1.5707676173343261 1.5707684608903676
Norm Error
 33054.99657540966
Subspace Error
 7 0.041557746914094536 0.3961454661866247
Norm Error
 12210.027929774054
Subspace Error
 8 0.010059654374264653 0.06725816293901576
Norm Error
 10960.195576230317
Subspace Error
 7 0.010058277164810564 0.0671590603633027
Norm Error
 10959.860203205846
Subspace Error
 6 0.010056845328456536 0.0670733380150883
Norm Error
 10955.830792011895
Subspace Error
 5 0.010061585954272936 0.0

In [24]:
so[:K]

array([57128.10560151, 14563.22961611,  8568.04834109,  8419.40461359,
        5775.33197752])

In [25]:
Ss

array([57128.10560151, 14563.22961611,  8568.04834109,  8419.40461359,
        5775.33197752])

In [26]:
U


Tensor train with order    = 9, 
                  row_dims = [2, 2, 2, 2, 2, 2, 2, 2, 2], 
                  col_dims = [1, 5, 1, 1, 1, 1, 1, 1, 1], 
                  ranks    = [1, 2, 20, 40, 32, 16, 8, 4, 2, 1]

In [27]:
uo[:,:K]

array([[-0.03631081, -0.04821844, -0.05238285, -0.02044104,  0.02528985],
       [-0.0202786 , -0.02693655, -0.02925368,  0.04203775,  0.01411369],
       [-0.04222002, -0.05618515, -0.06090737, -0.02905427,  0.02942966],
       ...,
       [-0.03054129,  0.01939687,  0.02751477,  0.06375176, -0.00908162],
       [-0.06042607,  0.03601381,  0.05443596, -0.04230555, -0.01793824],
       [-0.04217162,  0.02451796,  0.03799252,  0.05548045, -0.01251134]])

In [28]:
U.matricize()

array([[ 0.03631081, -0.04821844,  0.05238285, -0.02044104, -0.02528985],
       [ 0.0202786 , -0.02693655,  0.02925368,  0.04203775, -0.01411369],
       [ 0.04222002, -0.05618515,  0.06090737, -0.02905427, -0.02942966],
       ...,
       [ 0.03054129,  0.01939687, -0.02751477,  0.06375176,  0.00908162],
       [ 0.06042607,  0.03601381, -0.05443596, -0.04230555,  0.01793824],
       [ 0.04217162,  0.02451796, -0.03799252,  0.05548045,  0.01251134]])

In [29]:
V


Tensor train with order    = 9, 
                  row_dims = [3, 3, 3, 3, 3, 3, 3, 3, 3], 
                  col_dims = [1, 5, 1, 1, 1, 1, 1, 1, 1], 
                  ranks    = [1, 3, 45, 50, 50, 50, 27, 9, 3, 1]

In [30]:
V.matricize()

array([[ 0.00611295,  0.00731594,  0.00479314,  0.00017593,  0.00759757],
       [ 0.0036245 ,  0.00434997,  0.00284184,  0.00580662,  0.00450507],
       [ 0.00578396,  0.00692911,  0.00453515, -0.00266631,  0.00718837],
       ...,
       [ 0.0063546 , -0.00045078, -0.00886864,  0.00025783, -0.00809808],
       [ 0.00387809, -0.00030315, -0.00541249,  0.00755813, -0.00494226],
       [ 0.00628164, -0.00038147, -0.00876646, -0.00295635, -0.00800329]])

In [31]:
vo[:K, :].T

array([[-0.00611295,  0.00731594, -0.00479314,  0.00017593, -0.00759757],
       [-0.0036245 ,  0.00434997, -0.00284184,  0.00580662, -0.00450507],
       [-0.00578396,  0.00692911, -0.00453515, -0.00266631, -0.00718837],
       ...,
       [-0.0063546 , -0.00045078,  0.00886864,  0.00025783,  0.00809808],
       [-0.00387809, -0.00030315,  0.00541249,  0.00755813,  0.00494226],
       [-0.00628164, -0.00038147,  0.00876646, -0.00295635,  0.00800329]])